# Population Profiling
Owner: Rhea  
Table: `pondolens.dev_bronze.population`  
Baseline: 16,416 rows

**Goal:** identify population records suitable for Q3 and required Silver treatments. All queries are read-only.

**Row meaning:** geographic area × year/reference date × age group × sex. Values are mid-year population projections in persons.


## 1. Structure and coverage
Inspect columns, sample records, years, and geographic labels. PHILIPPINES is a national total, not a region.


In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.population;

In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.population
LIMIT 20;

In [0]:
%sql
SELECT
    year,
    COUNT(*) AS row_count,
    COUNT(DISTINCT region_name) AS geographic_label_count,
    SORT_ARRAY(COLLECT_SET(sex)) AS sex_labels,
    SORT_ARRAY(COLLECT_SET(reference_date)) AS reference_dates
FROM pondolens.dev_bronze.population
GROUP BY year
ORDER BY year;

In [0]:
%sql
SELECT
    region_name,
    COUNT(*) AS row_count,
    MIN(TRY_CAST(year AS INT)) AS first_year,
    MAX(TRY_CAST(year AS INT)) AS last_year
FROM pondolens.dev_bronze.population
GROUP BY region_name
ORDER BY region_name;

In [0]:
%sql
SELECT
    year,
    COUNT(DISTINCT region_name) AS all_geographic_labels,
    COUNT(DISTINCT CASE
        WHEN UPPER(TRIM(region_name)) <> 'PHILIPPINES'
        THEN region_name
    END) AS regions_only
FROM pondolens.dev_bronze.population
GROUP BY year
ORDER BY year;

## 2. Data quality
Check required values, numeric/date parsing, rescued data, and repeated candidate keys. Empty duplicate results mean no repeated keys were found.


In [0]:
%sql
WITH checked AS (
    SELECT *,
        TRY_CAST(population AS DECIMAL(20, 2)) AS population_value,
        TRY_CAST(year AS INT) AS year_value,
        TRY_CAST(reference_date AS DATE) AS reference_date_value
    FROM pondolens.dev_bronze.population
)
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(NULLIF(TRIM(region_name), '') IS NULL) AS missing_region,
    COUNT_IF(NULLIF(TRIM(age_group), '') IS NULL) AS missing_age_group,
    COUNT_IF(NULLIF(TRIM(sex), '') IS NULL) AS missing_sex,
    COUNT_IF(NULLIF(TRIM(population), '') IS NULL) AS missing_population,
    COUNT_IF(
        NULLIF(TRIM(population), '') IS NOT NULL
        AND population_value IS NULL
    ) AS invalid_population,
    COUNT_IF(population_value < 0) AS negative_population,
    COUNT_IF(population_value = 0) AS zero_population,
    COUNT_IF(population_value <> FLOOR(population_value)) AS fractional_population,
    COUNT_IF(year_value IS NULL) AS missing_or_invalid_year,
    COUNT_IF(reference_date_value IS NULL) AS missing_or_invalid_date,
    COUNT_IF(year_value <> YEAR(reference_date_value)) AS year_date_mismatch,
    COUNT_IF(NULLIF(TRIM(source), '') IS NULL) AS missing_source,
    COUNT_IF(NULLIF(TRIM(_rescued_data), '') IS NOT NULL) AS rescued_rows
FROM checked;

In [0]:
%sql
SELECT
    region_name,
    year,
    reference_date,
    age_group,
    sex,
    COUNT(*) AS row_count,
    COUNT(DISTINCT population) AS distinct_population_values,
    COUNT(DISTINCT source) AS distinct_sources
FROM pondolens.dev_bronze.population
GROUP BY region_name, year, reference_date, age_group, sex
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

## 3. Reconcile totals
Compare Both with Male + Female, all ages with age groups, and the national total with regional totals. Differences are findings; do not adjust source values to force agreement.


In [0]:
%sql
WITH totals AS (
    SELECT
        region_name,
        year,
        reference_date,
        age_group,
        SUM(CASE WHEN sex = 'Both'
            THEN TRY_CAST(population AS BIGINT) END) AS both_population,
        SUM(CASE WHEN sex = 'Male'
            THEN TRY_CAST(population AS BIGINT) END) AS male_population,
        SUM(CASE WHEN sex = 'Female'
            THEN TRY_CAST(population AS BIGINT) END) AS female_population,
        COUNT_IF(sex = 'Both') AS both_rows,
        COUNT_IF(sex = 'Male') AS male_rows,
        COUNT_IF(sex = 'Female') AS female_rows
    FROM pondolens.dev_bronze.population
    GROUP BY region_name, year, reference_date, age_group
)
SELECT *,
    both_population - (male_population + female_population) AS difference
FROM totals
WHERE both_rows <> 1
   OR male_rows <> 1
   OR female_rows <> 1
   OR both_population <> male_population + female_population
ORDER BY year, region_name, age_group;

In [0]:
%sql
WITH totals AS (
    SELECT
        region_name,
        year,
        reference_date,
        sex,
        COUNT_IF(age_group = 'Total') AS total_rows,
        COUNT_IF(age_group <> 'Total') AS age_group_rows,
        SUM(CASE WHEN age_group = 'Total'
            THEN TRY_CAST(population AS BIGINT) END) AS reported_total,
        SUM(CASE WHEN age_group <> 'Total'
            THEN TRY_CAST(population AS BIGINT) END) AS age_group_sum
    FROM pondolens.dev_bronze.population
    GROUP BY region_name, year, reference_date, sex
)
SELECT *,
    reported_total - age_group_sum AS difference
FROM totals
WHERE total_rows <> 1
   OR age_group_rows <> 18
   OR reported_total <> age_group_sum
ORDER BY year, region_name, sex;

In [0]:
%sql
SELECT
    year,
    SUM(CASE WHEN UPPER(TRIM(region_name)) = 'PHILIPPINES'
        THEN TRY_CAST(population AS BIGINT) END) AS national_population,
    SUM(CASE WHEN UPPER(TRIM(region_name)) <> 'PHILIPPINES'
        THEN TRY_CAST(population AS BIGINT) END) AS regional_population_sum,
    national_population - regional_population_sum AS difference
FROM pondolens.dev_bronze.population
WHERE age_group = 'Total'
  AND sex = 'Both'
GROUP BY year
ORDER BY year;

## 4. Q3 candidates and geographic compatibility
The population query includes PHILIPPINES for inspection. Regional denominators must exclude it. Compare GRDP labels before joining; matching names do not prove matching boundaries.


In [0]:
%sql
SELECT
    region_name,
    year,
    reference_date,
    population,
    source
FROM pondolens.dev_bronze.population
WHERE LOWER(TRIM(age_group)) = 'total'
  AND LOWER(TRIM(sex)) = 'both'
  AND TRY_CAST(year AS INT) IN (2024, 2025)
ORDER BY region_name, year;

In [0]:
%sql
SELECT DISTINCT Region
FROM pondolens.dev_bronze.grdp
ORDER BY Region;

## 5. Findings and Silver treatments

| Check | Finding | Silver treatment |
|---|---|---|
| Coverage | 16,416 rows; 2020–2035; 1,026 rows per year | Preserve all years; filter 2024–2025 for current analysis. |
| Required values | Checked fields have no missing/invalid values; no negative, zero, or fractional population; no rescued data | Cast population to BIGINT, year to INT, and reference_date to DATE. |
| Candidate keys | No repeated region/year/reference_date/age_group/sex combinations | Validate key uniqueness in Silver. |
| Age bounds | Sample Total rows have null bounds; 85 and over has no upper bound | Preserve meaningful nulls; validate remaining age bounds separately. |
| Sex totals | 1,295 groups differ by ±100 persons | Preserve reported values; verify source explanation. |
| Age totals | 589 groups differ; maximum absolute difference is 400 persons | Preserve reported totals; source confirms age-group rounding differences. |
| National total | 9 of 16 years differ from regional sums; maximum absolute difference is 200 persons | Keep national and regional totals separate; verify source explanation. |
| Geographic coverage | Original population workbook has 17 regional groups plus PHILIPPINES; GRDP has 18 regions plus Philippines, including NIR | Classify geographic level; resolve boundary compatibility before joining. |
| Q3 candidates | 34 regional Total/Both records for 2024–2025 | Use only after confirming compatible geographic coverage. |

Reconciliation counts describe groups across 2020–2035, not individual rows.

**Source evidence:** original population workbook, Table 1: regional headings in A27–A347; source in A368. The note in A367 confirms age-group estimates may not sum to all-age totals due to rounding. It does not explicitly explain sex or national/regional differences.

## 6. Assumptions and open checks

| Question | Status |
|---|---|
| Do source values explain sex and national/regional differences? | Pending sample reconciliation and methodology review. |
| How are NIR areas represented in the population workbook? | Pending source boundary confirmation; no separate NIR entry. |
| Are population, GRDP, and spending boundaries compatible? | Pending. GRDP includes NIR; spending coverage still needs checking. |
| Is July 1 projected population the agreed annual denominator? | Pending team agreement. |
| Are all age bounds valid and consistent with age-group labels? | Pending full-table check. |

## 7. Semantic notes

- Population means projected persons as of July 1, not an actual census count for every projection year.
- Regional denominator: Total age group + Both sexes, excluding PHILIPPINES.
- Do not sum totals with age details, Both with Male/Female, or national with regional records.
- Spending per person = comparable regional spending in PHP ÷ matching projected population. Return null and flag missing/zero denominators.
- Match year and verified geographic boundaries before joining.
- Population does not measure service access. Spending per person is not cash received by each resident.

## 8. Conclusion

Basic required-value, parsing, and candidate-key checks passed. Silver needs typed columns, geographic classification, and reconciliation flags while preserving source values.

Population profiling is still open for age-bound validation, source reconciliation, and geographic compatibility. The 34 regional records for 2024–2025 are Q3 candidates, not yet approved denominators for cross-dataset comparisons.
